In [3]:
# ЭТАП W1. Загрузка данных (только по 26.07.2026 включительно) и матрицы «серия × день»
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CANDIDATES = [
    ROOT / "data" / "processed" / "base_grid.csv",
    ROOT / "data" / "raw" / "base_grid.csv",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Не найден base_grid.csv. Положи его в data/processed/")

CUT_DATE = pd.Timestamp("2026-07-26")   # последний день, который разрешено использовать (воскресенье)
WEEKS = 9
HORIZON = WEEKS * 7                     # 63 дня
t0 = time.time()

dtypes = {
    "warehouse_name": "category", "article": "category",
    "stock": "float32", "quantity": "float32",
    "cities_name": "category",
    "category_lvl_1": "category", "category_lvl_2": "category", "category_lvl_3": "category",
}
df = pd.read_csv(DATA_PATH, dtype=dtypes, parse_dates=["date"])
df = df[df["date"] <= CUT_DATE]                       # данные после 26.07 использовать нельзя
df = df.sort_values(["warehouse_name", "article", "date"]).reset_index(drop=True)

KEYS = ["warehouse_name", "article"]
df["sid"] = df.groupby(KEYS, observed=True).ngroup()
S = df["sid"].nunique()

dates = pd.date_range(df["date"].min(), df["date"].max())
D = len(dates)
assert dates[-1] == CUT_DATE and CUT_DATE.weekday() == 6, "Данные должны заканчиваться воскресеньем 2026-07-26"
day_idx = (df["date"] - dates[0]).dt.days.values

Q = np.full((S, D), np.nan, dtype=np.float32)
Q[df["sid"].values, day_idx] = df["quantity"].values
ST = np.full((S, D), np.nan, dtype=np.float32)
ST[df["sid"].values, day_idx] = df["stock"].values

Q0 = np.nan_to_num(Q)
first_idx = np.argmax(~np.isnan(Q), axis=1)

meta = df.drop_duplicates("sid").set_index("sid").sort_index()
art_cat = (
    df[df["category_lvl_1"] != "unknown"]
    .drop_duplicates("article")
    .set_index("article")["category_lvl_1"]
)
meta["cat1"] = meta["article"].map(art_cat).astype("object").fillna("unknown")
wh_code = meta["warehouse_name"].cat.codes.values
cat_code = pd.Categorical(meta["cat1"]).codes
del df

print(f"Файл: {DATA_PATH}")
print(f"Серий: {S}, дней: {D}, период: {dates[0].date()} — {dates[-1].date()} ({dates[-1].day_name()})")
print(f"Прогноз: {WEEKS} недель, с {(dates[-1] + pd.Timedelta(days=1)).date()} "
      f"по {(dates[-1] + pd.Timedelta(days=HORIZON)).date()}")
print(f"Отрицательных остатков: {int((ST < 0).sum())}, пропусков остатка: {np.isnan(ST).mean():.1%}")
print(f"Готово за {time.time() - t0:.1f} c")

Файл: C:\Users\nikit\sber-forecast\data\processed\base_grid.csv
Серий: 8789, дней: 957, период: 2023-12-13 — 2026-07-26 (Sunday)
Прогноз: 9 недель, с 2026-07-27 по 2026-09-27
Отрицательных остатков: 29, пропусков остатка: 42.2%
Готово за 57.7 c


In [4]:
# ЭТАП W1b. Пересборка матрицы остатков ST по правилу ТЗ (запускать СРАЗУ ПОСЛЕ W1, до W3/W4)
# В base_grid.csv остатки собраны с drop_duplicates(date, склад, артикул, stock), а ТЗ требует суммировать ВСЕ строки.
raw_stock_p = ROOT / "data" / "raw" / "stock_dataset_hakaton.csv"
raw = pd.read_csv(raw_stock_p)
raw.columns = [str(c).strip().lstrip("\ufeff") for c in raw.columns]
if "char_article" in raw.columns and "article" not in raw.columns:
    raw = raw.rename(columns={"char_article": "article"})
raw["date"] = pd.to_datetime(raw["date"])
raw = raw[raw["date"] <= CUT_DATE]

series_idx = pd.MultiIndex.from_arrays([meta["warehouse_name"].astype(str).values, meta["article"].astype(str).values])


def build_stock_matrix(frame):
    """frame: date, warehouse_name, article, stock; строки с одним ключом уже агрегированы суммой."""
    g = frame.groupby(["warehouse_name", "article", "date"], as_index=False)["stock"].sum()
    g["warehouse_name"], g["article"] = g["warehouse_name"].astype(str), g["article"].astype(str)
    rows = series_idx.get_indexer(pd.MultiIndex.from_frame(g[["warehouse_name", "article"]]))
    cols = (g["date"] - dates[0]).dt.days.values
    ok = (rows >= 0) & (cols >= 0) & (cols < D)
    M = np.full((S, D), np.nan, dtype=np.float32)
    M[rows[ok], cols[ok]] = g["stock"].values[ok]
    return pd.DataFrame(M.T).ffill().to_numpy().T.astype(np.float32)     # как в base_grid: ffill, NaN только до первого остатка


# 1) Контроль: способ участника (drop_duplicates) должен воспроизвести ST из base_grid.csv
ST_dedup = build_stock_matrix(raw.drop_duplicates(subset=["date", "warehouse_name", "article", "stock"]))
same_nan = np.array_equal(np.isnan(ST_dedup), np.isnan(ST))
both = ~np.isnan(ST_dedup) & ~np.isnan(ST)
print(f"Контроль: способ с drop_duplicates воспроизводит base_grid: форма NaN совпадает = {same_nan}, "
      f"ячеек с расхождением значений = {int((ST_dedup[both] != ST[both]).sum())}")

# 2) Правильный способ по ТЗ: суммируем все строки
ST_sum = build_stock_matrix(raw)
print(f"По ТЗ (сумма всех строк): ячеек, где остаток изменился: "
      f"{int((ST_sum[both] != ST[both]).sum()):,}; NaN-маска совпадает с base_grid: {np.array_equal(np.isnan(ST_sum), np.isnan(ST))}")

ST = ST_sum
print("ST заменена на версию по ТЗ. Теперь запускай W3, W4 (и W6/W7, если нужны), затем W9-W11.")


Контроль: способ с drop_duplicates воспроизводит base_grid: форма NaN совпадает = True, ячеек с расхождением значений = 0
По ТЗ (сумма всех строк): ячеек, где остаток изменился: 15,806; NaN-маска совпадает с base_grid: True
ST заменена на версию по ТЗ. Теперь запускай W3, W4 (и W6/W7, если нужны), затем W9-W11.


In [5]:
# ЭТАП W2. Метрики по новому score.py, недельные окна, базовые прогнозы
def compute_metrics(Y, P):
    """Y, P: (серии x 9 недель). SMAPE: пары 0/0 остаются в среднем с ошибкой 0 (новый score.py)."""
    P = np.clip(P, 0, None)
    den = (np.abs(Y) + np.abs(P)) / 2
    ratio = np.zeros_like(den, dtype=float)
    np.divide(np.abs(Y - P), den, out=ratio, where=den > 0)
    smape = ratio.mean(axis=1).mean() * 100
    denom = np.abs(Y).sum()
    num = np.abs(Y - P).sum()
    wape = (0.0 if num == 0 else 100.0) if denom == 0 else num / denom * 100
    rmse = np.sqrt(((Y - P) ** 2).mean())
    rmsle = np.sqrt(((np.log1p(P) - np.log1p(Y)) ** 2).mean())
    my, mp = Y.mean(axis=1), P.mean(axis=1)
    ok = my > 0
    bias = (np.abs(mp[ok] - my[ok]) / my[ok]).mean() * 100
    return {"SMAPE": smape, "WAPE": wape, "RMSE": rmse, "Bias": bias, "RMSLE": rmsle}


WEIGHTS = {"SMAPE": 0.50, "WAPE": 0.20, "RMSE": 0.15, "Bias": 0.10, "RMSLE": 0.05}


def proxy_score(m, naive):
    """Ориентир (не рейтинг): метрики, делённые на метрики «последней недели», с весами."""
    return sum(WEIGHTS[k] * min(m[k] / naive[k], 1.0) for k in WEIGHTS)


def window_cut(k):
    """Последний известный день окна проверки k (воскресенье), после него идут 9 недель факта."""
    return D - 1 - HORIZON * (k + 1)


def truth(c):
    """Факт: недельные суммы за 9 недель после дня c -> (серии x 9)."""
    return Q0[:, c + 1:c + 1 + HORIZON].reshape(S, WEEKS, 7).sum(axis=2)


def flat_w(rate):
    """Одно число на серию -> плоский прогноз на 9 недель."""
    return np.repeat(np.asarray(rate, dtype=np.float32)[:, None], WEEKS, axis=1)


def baselines(c):
    last4 = np.stack([Q0[:, c - 7 * (i + 1) + 1:c - 7 * i + 1].sum(axis=1) for i in range(4)], axis=1)
    m13 = Q0[:, c - 90:c + 1].sum(axis=1) / 13
    return {
        "нули": flat_w(np.zeros(S)),
        "последняя неделя": flat_w(last4[:, 0]),
        "среднее 4 нед.": flat_w(last4.mean(axis=1)),
        "медиана 4 нед.": flat_w(np.median(last4, axis=1)),
        "среднее 13 нед.": flat_w(m13),
        "среднее 13 нед., ноль если < 0.5": flat_w(np.where(m13 < 0.5, 0, m13)),
    }


N_WINDOWS = 3
naive_m, truths, rows = {}, {}, []
for k in range(N_WINDOWS):
    c = window_cut(k)
    truths[k] = truth(c)
    naive_m[k] = compute_metrics(truths[k], baselines(c)["последняя неделя"])
    for name, P in baselines(c).items():
        m = compute_metrics(truths[k], P)
        rows.append({"подход": name, **m, "score": proxy_score(m, naive_m[k])})

summary = pd.DataFrame(rows).groupby("подход").mean().round(3).sort_values("score")
for k in range(N_WINDOWS):
    c = window_cut(k)
    print(f"окно {k}: до {dates[c].date()}, факт {dates[c + 1].date()} — {dates[c + HORIZON].date()}, "
          f"серий без продаж за 9 недель: {(truths[k].sum(1) == 0).sum()}")
display(summary)

окно 0: до 2026-05-24, факт 2026-05-25 — 2026-07-26, серий без продаж за 9 недель: 1058
окно 1: до 2026-03-22, факт 2026-03-23 — 2026-05-24, серий без продаж за 9 недель: 978
окно 2: до 2026-01-18, факт 2026-01-19 — 2026-03-22, серий без продаж за 9 недель: 877


,SMAPE,WAPE,RMSE,Bias,RMSLE,score
подход,,,,,,
"среднее 13 нед., ноль если < 0.5",70.609,50.412998,2.880,75.769997,0.505,0.857
среднее 13 нед.,123.279,52.296001,2.873,63.303001,0.492,0.892
медиана 4 нед.,75.086,51.394001,2.907,81.416000,0.529,0.898
среднее 4 нед.,98.802,52.234001,2.850,78.639000,0.514,0.905
нули,97.498,100.000000,7.319,100.000000,1.094,0.982
последняя неделя,76.778,60.660999,3.357,122.698997,0.623,1.000


In [6]:
# ЭТАП W3. Недельная модель: признаки, обучающая выборка, параметры
import importlib
import lightgbm as lgb

# новое ТЗ: отрицательный остаток = резерв, то есть свободного товара нет -> считаем «нет товара» при остатке <= 0
fp = ROOT / "src" / "features.py"
txt = fp.read_text(encoding="utf-8")
if "(sw == 0)" in txt:
    fp.write_text(txt.replace("(sw == 0)", "(sw <= 0)"), encoding="utf-8")
from src import features as _features
importlib.reload(_features)

SEED = 42
N_JOBS = 4          # не меняй число потоков между запусками


def build_features(c):
    return _features.build_features(Q0, ST, c, first_idx, wh_code, cat_code)


def make_training_set(last_cut, step=7):
    """Цель: средние недельные продажи за 9 недель после даты отсечки cc (cc всегда воскресенье)."""
    parts_X, parts_y = [], []
    for cc in range(last_cut, 179, -step):
        parts_X.append(build_features(cc))
        parts_y.append(Q0[:, cc + 1:cc + 1 + HORIZON].sum(axis=1) / WEEKS)
    X = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X[col] = X[col].astype("category")
    return X, np.concatenate(parts_y)


PARAMS = dict(
    objective="tweedie", tweedie_variance_power=1.3,
    n_estimators=400, learning_rate=0.05, num_leaves=63, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=SEED, deterministic=True, force_row_wise=True, n_jobs=N_JOBS, verbose=-1,
)


def fit_rate(c_known):
    """Учим на датах, цель которых закончилась не позже c_known; прогноз недельной интенсивности на c_known."""
    X, y = make_training_set(c_known - HORIZON)
    model = lgb.LGBMRegressor(**PARAMS).fit(X, y)
    return model, np.clip(model.predict(build_features(c_known)), 0, None), len(X)


print("Определения готовы. Признаков:", build_features(D - 1).shape[1])

Определения готовы. Признаков: 22


In [7]:
# ЭТАП W4. Финальное обучение и submission (9 недель, первая неделя 2026-07-27)
FINAL_THRESHOLD = 1.0      # недельный прогноз ниже порога превращается в 0
FINAL_SCALE = 1.0          # множитель на прогноз
CHECK_DETERMINISM = True

t0 = time.time()
c_final = D - 1                                    # воскресенье 2026-07-26
X_fin, y_fin = make_training_set(c_final - HORIZON)
X_now = build_features(c_final)
print(f"Обучающая выборка: {len(X_fin):,} строк, подготовка за {time.time() - t0:.0f} c")


def train_and_predict():
    model = lgb.LGBMRegressor(**PARAMS).fit(X_fin, y_fin)
    return model, np.clip(model.predict(X_now), 0, None)


t1 = time.time()
final_model, rate_final = train_and_predict()
print(f"Модель обучена за {time.time() - t1:.0f} c")
if CHECK_DETERMINISM:
    _, rate_again = train_and_predict()
    print("Детерминизм (два обучения дали одинаковый результат):",
          "ДА" if np.array_equal(rate_final, rate_again) else "НЕТ")

rate_post = rate_final * FINAL_SCALE
rate_post = np.where(rate_post < FINAL_THRESHOLD, 0.0, rate_post)

weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")   # понедельники
sub = pd.DataFrame({
    "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
    "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
    "article": np.repeat(meta["article"].astype(str).values, WEEKS),
    "quantity": np.round(np.repeat(rate_post, WEEKS), 4),
})


def check_submission(df):
    problems = []
    if set(["week", "warehouse_name", "article", "quantity"]) - set(df.columns):
        return ["нет нужных колонок"]
    if df["quantity"].isna().any():
        problems.append("есть NaN в quantity")
    if (df["quantity"] < 0).any():
        problems.append("есть отрицательные значения")
    wk = pd.to_datetime(df["week"])
    if not (wk.dt.weekday == 0).all():
        problems.append("не все недели начинаются с понедельника")
    uw = np.sort(wk.unique())
    if len(uw) != WEEKS:
        problems.append(f"недель {len(uw)}, ожидалось {WEEKS}")
    elif (pd.Series(uw).diff().dropna() != pd.Timedelta(days=7)).any():
        problems.append("недели идут не подряд")
    if str(pd.Timestamp(uw[0]).date()) != "2026-07-27":
        problems.append("первая неделя не 2026-07-27")
    if df.duplicated(["week", "warehouse_name", "article"]).any():
        problems.append("есть дубли ключа")
    if len(df) != S * WEEKS:
        problems.append(f"строк {len(df)}, ожидалось {S * WEEKS}")
    return problems


problems = check_submission(sub)
print("\nПроверка формата:", "ВСЁ В ПОРЯДКЕ" if not problems else problems)
last9 = Q0[:, c_final - HORIZON + 1:c_final + 1].sum()
print(f"Прогноз за 9 недель: {sub['quantity'].sum():,.0f} шт.; факт предыдущих 9 недель: {last9:,.0f} шт.; "
      f"отношение {sub['quantity'].sum() / last9:.2f}")
print(f"Серий, которым прогноз обнулён: {(rate_post == 0).sum()} из {S} ({(rate_post == 0).mean():.0%})")
print(f"Недели прогноза: {weeks[0].date()} — {weeks[-1].date()} ({len(sub):,} строк)")

if not problems:
    out_path = ROOT / "data" / "processed" / "submission_weekly.csv"
    sub.to_csv(out_path, index=False)
    print(f"\nФайл сохранён: {out_path} ({out_path.stat().st_size / 1e6:.1f} МБ)")
    display(sub.head(3))
else:
    print("\nФайл НЕ сохранён: исправь проблемы выше.")

Обучающая выборка: 896,478 строк, подготовка за 14 c
Модель обучена за 60 c
Детерминизм (два обучения дали одинаковый результат): ДА

Проверка формата: ВСЁ В ПОРЯДКЕ
Прогноз за 9 недель: 209,259 шт.; факт предыдущих 9 недель: 212,336 шт.; отношение 0.99
Серий, которым прогноз обнулён: 5198 из 8789 (59%)
Недели прогноза: 2026-07-27 — 2026-09-21 (79,101 строк)

Файл сохранён: C:\Users\nikit\sber-forecast\data\processed\submission_weekly.csv (4.1 МБ)


,week,warehouse_name,article,quantity
0,2026-07-27,warehouse_name_0,char_article_0,0.0
1,2026-08-03,warehouse_name_0,char_article_0,0.0
2,2026-08-10,warehouse_name_0,char_article_0,0.0


In [8]:
# ЭТАП W5. Проверка порога и масштаба на трёх окнах и на сезонном окне (аналог 27.07.2025)
c_s = int((pd.Timestamp("2025-07-27") - dates[0]).days)
truths_s = truth(c_s)
naive_s = compute_metrics(truths_s, baselines(c_s)["последняя неделя"])

rates, t0 = {}, time.time()
for k in range(N_WINDOWS):
    _, rates[k], n = fit_rate(window_cut(k))
    print(f"окно {k}: {n:,} строк, {time.time() - t0:.0f} c", flush=True)
_, rate_s, n = fit_rate(c_s)
print(f"сезонное: {n:,} строк, {time.time() - t0:.0f} c", flush=True)

TH = [0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
SC = [0.9, 1.0, 1.1, 1.2]
combos = [(a, b) for a in TH for b in SC]


def sc_for(rate, Y, naive):
    out = {}
    for th, s in combos:
        r = np.where(rate * s < th, 0.0, rate * s)
        out[(th, s)] = proxy_score(compute_metrics(Y, flat_w(r)), naive)
    return out


tab = {k: sc_for(rates[k], truths[k], naive_m[k]) for k in range(N_WINDOWS)}
honest = []
for held in range(N_WINDOWS):
    oth = [k for k in range(N_WINDOWS) if k != held]
    best = min(combos, key=lambda cb: np.mean([tab[k][cb] for k in oth]))
    honest.append(tab[held][best])
best_all = min(combos, key=lambda cb: np.mean([tab[k][cb] for k in range(N_WINDOWS)]))

print("\nМодель без обработки:", round(np.mean([tab[k][(0, 1.0)] for k in range(N_WINDOWS)]), 4))
print(f"Лучший порог/масштаб по 3 окнам: {best_all}, score "
      f"{np.mean([tab[k][best_all] for k in range(N_WINDOWS)]):.4f}; честный {np.mean(honest):.4f}")
print("\nscore по сочетаниям (среднее по 3 окнам), строки — порог, столбцы — масштаб:")
display(pd.DataFrame({s: [np.mean([tab[k][(t, s)] for k in range(N_WINDOWS)]) for t in TH] for s in SC},
                     index=TH).round(4))

ts = sc_for(rate_s, truths_s, naive_s)
print("\nСЕЗОННОЕ окно (авг–сен 2025): прогноз/факт по объёму:", round(rate_s.sum() * WEEKS / truths_s.sum(), 3))
display(pd.DataFrame({s: [ts[(t, s)] for t in TH] for s in SC}, index=TH).round(4))
for k in range(N_WINDOWS):
    print(f"окно {k} прогноз/факт по объёму:", round(rates[k].sum() * WEEKS / truths[k].sum(), 3))

окно 0: 817,377 строк, 69 c
окно 1: 738,276 строк, 134 c
окно 2: 659,175 строк, 192 c
сезонное: 439,450 строк, 236 c

Модель без обработки: 0.8975
Лучший порог/масштаб по 3 окнам: (1.0, 0.9), score 0.7931; честный 0.8016

score по сочетаниям (среднее по 3 окнам), строки — порог, столбцы — масштаб:


,0.9,1.0,1.1,1.2
0.00,0.8857,0.8975,0.9233,0.9460
0.25,0.8889,0.8996,0.9247,0.9480
0.50,0.8472,0.8705,0.9114,0.9453
0.75,0.7990,0.8113,0.8419,0.8767
1.00,0.7931,0.7984,0.8208,0.8498
1.50,0.8177,0.8125,0.8263,0.8445
2.00,0.8544,0.8437,0.8512,0.8615
3.00,0.9184,0.9036,0.9047,0.9083



СЕЗОННОЕ окно (авг–сен 2025): прогноз/факт по объёму: 0.857


,0.9,1.0,1.1,1.2
0.00,0.9103,0.9029,0.9066,0.9214
0.25,0.9181,0.9092,0.9116,0.9252
0.50,0.8776,0.8733,0.8869,0.9142
0.75,0.8449,0.8330,0.8344,0.8494
1.00,0.8489,0.8313,0.8253,0.8354
1.50,0.8783,0.8551,0.8436,0.8456
2.00,0.9165,0.8873,0.8705,0.8687
3.00,0.9741,0.9494,0.9307,0.9251


окно 0 прогноз/факт по объёму: 0.906
окно 1 прогноз/факт по объёму: 1.037
окно 2 прогноз/факт по объёму: 1.139


In [9]:
# ЭТАП W6. Вариант «медиана»: модель предсказывает типичную (медианную) неделю, а не среднюю
MEDIAN_THRESHOLD = 1.0     # недельный прогноз ниже порога превращается в 0
MEDIAN_SCALE = 1.1         # медианная модель слегка занижает, поэтому множитель 1.1

c_final = D - 1                                    # воскресенье 2026-07-26


def make_training_set_median(last_cut, step=7):
    """Цель: медиана недельных продаж за 9 недель после даты отсечки (для редких товаров это 0)."""
    parts_X, parts_y = [], []
    for cc in range(last_cut, 179, -step):
        parts_X.append(build_features(cc))
        weekly = Q0[:, cc + 1:cc + 1 + HORIZON].reshape(S, WEEKS, 7).sum(axis=2)
        parts_y.append(np.median(weekly, axis=1))
    X = pd.concat(parts_X, ignore_index=True)
    for col in ("warehouse", "category"):
        X[col] = X[col].astype("category")
    return X, np.concatenate(parts_y)


PARAMS_MED = {k: v for k, v in PARAMS.items() if k != "tweedie_variance_power"}
PARAMS_MED["objective"] = "l1"                     # L1 = модель ищет медиану

t0 = time.time()
X_med, y_med = make_training_set_median(c_final - HORIZON)
X_now_med = build_features(c_final)
print(f"Обучающая выборка: {len(X_med):,} строк, подготовка за {time.time() - t0:.0f} c")


def train_and_predict_median():
    model = lgb.LGBMRegressor(**PARAMS_MED).fit(X_med, y_med)
    return model, np.clip(model.predict(X_now_med), 0, None)


t1 = time.time()
model_med, rate_med = train_and_predict_median()
print(f"Модель обучена за {time.time() - t1:.0f} c")
_, rate_med_again = train_and_predict_median()
print("Детерминизм (два обучения дали одинаковый результат):",
      "ДА" if np.array_equal(rate_med, rate_med_again) else "НЕТ")

rate_med_post = rate_med * MEDIAN_SCALE
rate_med_post = np.where(rate_med_post < MEDIAN_THRESHOLD, 0.0, rate_med_post)

weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")
sub_med = pd.DataFrame({
    "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
    "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
    "article": np.repeat(meta["article"].astype(str).values, WEEKS),
    "quantity": np.round(np.repeat(rate_med_post, WEEKS), 4),
})

problems = check_submission(sub_med)            # функция из ячейки W4
print("\nПроверка формата:", "ВСЁ В ПОРЯДКЕ" if not problems else problems)
last9 = Q0[:, c_final - HORIZON + 1:c_final + 1].sum()
print(f"Прогноз за 9 недель: {sub_med['quantity'].sum():,.0f} шт.; факт предыдущих 9 недель: {last9:,.0f} шт.; "
      f"отношение {sub_med['quantity'].sum() / last9:.2f}")
print(f"Серий, которым прогноз обнулён: {(rate_med_post == 0).sum()} из {S} ({(rate_med_post == 0).mean():.0%})")

# сравнение с вариантом из W4 (средняя модель)
zero_a, zero_b = (rate_post == 0), (rate_med_post == 0)
print(f"Совпадение с вариантом W4: обнулены в обоих {int((zero_a & zero_b).sum())}, "
      f"только в W4 {int((zero_a & ~zero_b).sum())}, только в W6 {int((~zero_a & zero_b).sum())}")

if not problems:
    out_path = ROOT / "data" / "processed" / "submission_median.csv"
    sub_med.to_csv(out_path, index=False)
    print(f"\nФайл сохранён: {out_path} ({out_path.stat().st_size / 1e6:.1f} МБ)")
else:
    print("\nФайл НЕ сохранён: исправь проблемы выше.")

Обучающая выборка: 896,478 строк, подготовка за 17 c
Модель обучена за 75 c
Детерминизм (два обучения дали одинаковый результат): ДА

Проверка формата: ВСЁ В ПОРЯДКЕ
Прогноз за 9 недель: 210,043 шт.; факт предыдущих 9 недель: 212,336 шт.; отношение 0.99
Серий, которым прогноз обнулён: 5193 из 8789 (59%)
Совпадение с вариантом W4: обнулены в обоих 5074, только в W4 124, только в W6 119

Файл сохранён: C:\Users\nikit\sber-forecast\data\processed\submission_median.csv (4.0 МБ)


In [10]:
# ЭТАП W7. Несколько вариантов прогноза из уже обученных моделей (для отправки организатору и сравнения по лидерборду)
VARIANTS = {
    # имя файла : (какая модель, порог, масштаб)
    "submission_A_mean_t1.0_s1.0":   ("mean",   1.0, 1.0),
    "submission_B_median_t1.0_s1.1": ("median", 1.0, 1.1),
    "submission_C_median_t0.5_s1.2": ("median", 0.5, 1.2),
    "submission_D_mean_t0.75_s1.1":  ("mean",   0.75, 1.1),
}
RATES = {"mean": rate_final, "median": rate_med}          # из ячеек W4 и W6
weeks = pd.date_range(dates[-1] + pd.Timedelta(days=1), periods=WEEKS, freq="7D")

summary_rows = []
for name, (kind, th, sc) in VARIANTS.items():
    r = RATES[kind] * sc
    r = np.where(r < th, 0.0, r)
    sub_v = pd.DataFrame({
        "week": np.tile(weeks.strftime("%Y-%m-%d"), S),
        "warehouse_name": np.repeat(meta["warehouse_name"].astype(str).values, WEEKS),
        "article": np.repeat(meta["article"].astype(str).values, WEEKS),
        "quantity": np.round(np.repeat(r, WEEKS), 4),
    })
    problems = check_submission(sub_v)                     # функция из ячейки W4
    if not problems:
        sub_v.to_csv(ROOT / "data" / "processed" / f"{name}.csv", index=False)
    summary_rows.append({
        "файл": name + ".csv", "формат": "ок" if not problems else str(problems),
        "объём за 9 нед.": round(float(sub_v["quantity"].sum())),
        "обнулено серий": f"{(r == 0).mean():.0%}",
    })
display(pd.DataFrame(summary_rows))

,файл,формат,объём за 9 нед.,обнулено серий
0,submission_A_mean_t1.0_s1.0.csv,ок,209259,59%
1,submission_B_median_t1.0_s1.1.csv,ок,210043,59%
2,submission_C_median_t0.5_s1.2.csv,ок,231885,55%
3,submission_D_mean_t0.75_s1.1.csv,ок,236917,50%


In [11]:
# ЭТАП W8. Записываем весь код контейнера: src/, app/, Dockerfile, requirements.txt, тест API
from importlib import metadata
from pathlib import Path

FILES = {
    "src/features.py": r'''"""Признаки для прогноза продаж. Единый источник кода для ноутбука и контейнера.

Правила: используются только данные до даты отсечки включительно, окна не длиннее 180 дней.
"""
import warnings

import numpy as np
import pandas as pd

WINDOW = 180
AGE_CAP = WINDOW - 1   # контейнер видит ровно 180 дней, поэтому возраст ряда ограничен 179

FEATURE_ORDER = [
    "mean_7", "mean_14", "mean_28", "mean_56", "mean_90", "mean_180",
    "share_pos_28", "share_pos_90", "share_pos_180", "max_90", "max_180",
    "days_since_sale", "stock_last", "stock_missing_last", "oos_share_30", "oos_share_90",
    "stock_missing_90", "stock_mean_90", "mean_in_stock_90", "age", "warehouse", "category",
]


def build_features(Q0, ST, c, first_idx, wh_code, cat_code):
    """
    Q0  — продажи (серии x дни), пропуски заменены нулями
    ST  — остатки (серии x дни), NaN = данных нет; отрицательный остаток = резерв (товара нет)
    c   — индекс последнего известного дня
    first_idx — индекс первого дня ряда, wh_code / cat_code — коды склада и категории
    """
    f = {}
    for w in (7, 14, 28, 56, 90, 180):
        f[f"mean_{w}"] = Q0[:, c - w + 1:c + 1].mean(axis=1)
    for w in (28, 90, 180):
        f[f"share_pos_{w}"] = (Q0[:, c - w + 1:c + 1] > 0).mean(axis=1)
    for w in (90, 180):
        f[f"max_{w}"] = Q0[:, c - w + 1:c + 1].max(axis=1)

    pos = Q0[:, c - 179:c + 1] > 0
    f["days_since_sale"] = np.where(pos.any(axis=1), 179 - np.argmax(pos[:, ::-1], axis=1), 180)

    s = ST[:, c - 179:c + 1]
    f["stock_last"] = s[:, -1]
    f["stock_missing_last"] = np.isnan(s[:, -1]).astype(np.float32)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for w in (30, 90):
            sw = s[:, -w:]
            zero_share = np.where(np.isnan(sw), np.nan, (sw <= 0).astype(np.float32))
            f[f"oos_share_{w}"] = np.nanmean(zero_share, axis=1)
        f["stock_missing_90"] = np.isnan(s[:, -90:]).mean(axis=1)
        f["stock_mean_90"] = np.nanmean(s[:, -90:], axis=1)

    q90 = Q0[:, c - 89:c + 1]
    in_stock = ST[:, c - 90:c] > 0
    cnt_in = in_stock.sum(axis=1)
    f["mean_in_stock_90"] = np.where(
        cnt_in > 0, (q90 * in_stock).sum(axis=1) / np.maximum(cnt_in, 1), np.nan)

    f["age"] = np.minimum(c - first_idx, AGE_CAP)
    f["warehouse"] = wh_code
    f["category"] = cat_code

    X = pd.DataFrame(f)[FEATURE_ORDER]
    X["warehouse"] = X["warehouse"].astype("category")
    X["category"] = X["category"].astype("category")
    return X
''',
    "src/predictor_core.py": r'''"""Предсказание по матрицам последних 180 дней (недельный прогноз).

Контракт: на вход матрицы продаж и остатков (серии x 180 дней, NaN = данных нет),
названия складов и артикулов каждой серии. На выходе матрица прогноза (серии x 9 недель):
плоский недельный прогноз с учётом порога обнуления и масштаба из config.json.
"""
import json
from pathlib import Path

import lightgbm as lgb
import numpy as np

from . import features


def load_artifacts(model_dir):
    model_dir = Path(model_dir)
    cfg = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
    booster = lgb.Booster(model_file=str(model_dir / "model.txt"))
    return {"cfg": cfg, "booster": booster}


def predict_from_arrays(Qw, STw, warehouses, articles, art):
    cfg = art["cfg"]
    window = cfg["window"]
    if Qw.shape[1] != window or STw.shape[1] != window:
        raise ValueError(f"Нужно ровно {window} дней, получено {Qw.shape[1]}")

    wh_index = {n: i for i, n in enumerate(cfg["warehouses"])}
    cat_index = {n: i for i, n in enumerate(cfg["categories"])}
    unknown = cat_index.get("unknown", -1)
    wh_code = np.array([wh_index.get(w, -1) for w in warehouses])
    cat_code = np.array([cat_index.get(cfg["article_category"].get(a, "unknown"), unknown)
                         for a in articles])

    Q0 = np.nan_to_num(Qw)
    first_idx = np.argmax(~np.isnan(Qw), axis=1)
    X = features.build_features(Q0, STw, window - 1, first_idx, wh_code, cat_code)

    rate = np.clip(art["booster"].predict(X), 0, None) * cfg["scale"]
    rate = np.where(rate < cfg["threshold"], 0.0, rate)
    return np.repeat(rate[:, None], cfg["weeks"], axis=1)
''',
    "src/raw_to_arrays.py": r'''"""Сборка матриц «серии x последние 180 дней» из сырых sales и stock (как приходят в API)."""
import numpy as np
import pandas as pd


def _fill(matrix, frame, idx, start):
    """Складываем значения frame (warehouse_name, article, date, value) в матрицу по индексам серий."""
    g = frame.groupby(["warehouse_name", "article", "date"], as_index=False)["value"].sum()
    rows = idx.get_indexer(pd.MultiIndex.from_frame(g[["warehouse_name", "article"]]))
    cols = (g["date"] - start).dt.days.values
    ok = rows >= 0                                  # серии, которых не было при обучении, пропускаем
    matrix[rows[ok], cols[ok]] = g["value"].values[ok]


def build_arrays(df_sales, df_stock, pairs, window=180):
    """
    df_sales — сырые продажи (несколько строк на день из-за composite_division суммируются)
    df_stock — сырые остатки (колонка char_article = article, дубли суммируются)
    pairs    — список пар [склад, артикул] в порядке серий модели
    Возвращает Qw, STw (серии x window) и последнюю дату в sales.
    """
    sales_dates = pd.to_datetime(df_sales["date"]).dt.normalize()
    last_date = sales_dates.max()
    start = last_date - pd.Timedelta(days=window - 1)
    idx = pd.MultiIndex.from_tuples([tuple(p) for p in pairs], names=["warehouse_name", "article"])

    s = pd.DataFrame({
        "warehouse_name": df_sales["warehouse_name"].astype(str).values,
        "article": df_sales["article"].astype(str).values,
        "date": sales_dates.values,
        "value": pd.to_numeric(df_sales["quantity"], errors="coerce").fillna(0.0).values,
    })
    s = s[(s["date"] >= start) & (s["date"] <= last_date)]
    Qw = np.zeros((len(idx), window), dtype=np.float32)          # дни без продаж = 0
    _fill(Qw, s, idx, start)

    k = pd.DataFrame({
        "warehouse_name": df_stock["warehouse_name"].astype(str).values,
        "article": df_stock["char_article"].astype(str).values,
        "date": pd.to_datetime(df_stock["date"]).dt.normalize().values,
        "value": pd.to_numeric(df_stock["stock"], errors="coerce").values,
    })
    k = k.dropna(subset=["value"])
    k = k[k["date"] <= last_date]                                # остатки позже последней даты sales не берём
    # Дубли (дата, склад, артикул) суммируем целиком, как требует ТЗ: без drop_duplicates.
    k = k.groupby(["warehouse_name", "article", "date"], as_index=False)["value"].sum()

    STw = np.full((len(idx), window), np.nan, dtype=np.float32)  # нет данных = NaN (остаток может быть отрицательным)
    _fill(STw, k[k["date"] >= start], idx, start)

    # Как в base_grid: остаток «тянется вперёд» от последнего известного значения (ffill),
    # NaN остаётся только до первого появления остатка. Если файл длиннее окна,
    # последнее значение до начала окна переносится в первый день окна.
    prev = (k[k["date"] < start].sort_values("date")
            .groupby(["warehouse_name", "article"], as_index=False).last())
    if len(prev):
        rows = idx.get_indexer(pd.MultiIndex.from_frame(prev[["warehouse_name", "article"]]))
        ok = (rows >= 0)
        rows, vals = rows[ok], prev["value"].values[ok]
        first_empty = np.isnan(STw[rows, 0])
        STw[rows[first_empty], 0] = vals[first_empty]
    STw = pd.DataFrame(STw.T).ffill().to_numpy().T.astype(np.float32)
    return Qw, STw, last_date
''',
    "app/predictor.py": r'''from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

from src import predictor_core, raw_to_arrays


class NotReadyError(RuntimeError):
    """Прогноз ещё не рассчитан (не было POST /predict)."""


class Predictor:
    """Обёртка над моделью для инференса."""

    def __init__(self, art):
        self.art = art
        self._cached_forecast: Optional[pd.DataFrame] = None

    @classmethod
    def load(cls, model_dir: Path) -> "Predictor":
        return cls(predictor_core.load_artifacts(model_dir))

    def predict(self, df_sales: pd.DataFrame, df_stock: pd.DataFrame) -> pd.DataFrame:
        """Принимает сырые sales и stock, возвращает прогноз на 9 недель в формате submission."""
        cfg = self.art["cfg"]
        pairs, weeks = cfg["pairs"], cfg["weeks"]
        Qw, STw, last_date = raw_to_arrays.build_arrays(df_sales, df_stock, pairs, cfg["window"])
        warehouses = [p[0] for p in pairs]
        articles = [p[1] for p in pairs]
        P = predictor_core.predict_from_arrays(Qw, STw, warehouses, articles, self.art)

        # прогноз начинается с понедельника календарной недели, следующей за неделей последней даты sales
        this_week_monday = last_date - pd.Timedelta(days=last_date.weekday())
        week_dates = pd.date_range(this_week_monday + pd.Timedelta(days=7), periods=weeks, freq="7D")
        df_forecast = pd.DataFrame({
            "week": np.tile(week_dates.strftime("%Y-%m-%d"), len(pairs)),
            "warehouse_name": np.repeat(warehouses, weeks),
            "article": np.repeat(articles, weeks),
            "quantity": np.round(P.reshape(-1), 4),
        })
        self._cached_forecast = df_forecast
        return df_forecast

    def get_cached_forecast(self) -> pd.DataFrame:
        if self._cached_forecast is None:
            raise NotReadyError("Прогноз не рассчитан. Сначала вызовите POST /predict.")
        return self._cached_forecast
''',
    "app/validators.py": r'''from typing import List

import pandas as pd

SALES_REQUIRED_COLS = [
    "date", "warehouse_name", "article", "quantity",
    "sales_sum", "discount_sum",
    "composite_division", "cities_name",
    "category_lvl_1", "category_lvl_2", "category_lvl_3",
    "characteristic",
]

STOCK_REQUIRED_COLS = ["date", "warehouse_name", "char_article", "stock"]

MIN_HISTORY_DAYS = 180


def validate_sales_csv(df: pd.DataFrame) -> List[str]:
    errors = []
    missing = set(SALES_REQUIRED_COLS) - set(df.columns)
    if missing:
        errors.append(f"Отсутствуют колонки в sales: {sorted(missing)}")
        return errors

    try:
        dates = pd.to_datetime(df["date"], format="%Y-%m-%d")
    except Exception:
        errors.append("Неверный формат даты в sales (ожидается YYYY-MM-DD)")
        return errors

    history_days = (dates.max() - dates.min()).days + 1
    if history_days < MIN_HISTORY_DAYS:
        errors.append(f"Недостаточно истории в sales: {history_days} дней, минимум {MIN_HISTORY_DAYS}")

    # Null в discount_sum означает отсутствие скидки и заменяется нулём.
    df["discount_sum"] = df["discount_sum"].fillna(0)

    quantity = pd.to_numeric(df["quantity"], errors="coerce")
    if quantity.isna().any():
        errors.append("В quantity есть нечисловые или пустые значения")
    elif (quantity < 0).any():
        errors.append("Найдены отрицательные значения в quantity")

    return errors


def validate_stock_csv(df: pd.DataFrame) -> List[str]:
    errors = []
    missing = set(STOCK_REQUIRED_COLS) - set(df.columns)
    if missing:
        errors.append(f"Отсутствуют колонки в stock: {sorted(missing)}")
        return errors

    try:
        dates = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise")
    except Exception:
        errors.append("Неверный формат даты в stock (ожидается YYYY-MM-DD)")
        return errors

    history_days = (dates.max() - dates.min()).days + 1
    if history_days < MIN_HISTORY_DAYS:
        errors.append(f"Недостаточно истории в stock: {history_days} дней, минимум {MIN_HISTORY_DAYS}")

    # Отрицательные значения stock допустимы: это зарезервированный товар.
    if pd.to_numeric(df["stock"], errors="coerce").isna().any():
        errors.append("В stock есть нечисловые или пустые значения")

    return errors
''',
    "app/main.py": r'''import io
from pathlib import Path
from typing import Optional

import pandas as pd
from fastapi import FastAPI, File, Query, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse, StreamingResponse

from .predictor import NotReadyError, Predictor
from .validators import validate_sales_csv, validate_stock_csv

app = FastAPI(
    title="Hackathon Forecast API",
    description="Прогноз продаж по складам и товарам (9 недель)",
    version="1.0.0",
)

# Модель загружается в память при старте контейнера
predictor = Predictor.load(Path("model"))


class ApiError(Exception):
    """Ошибка с единым форматом ответа: {"detail": ..., "code": ...}."""

    def __init__(self, status_code: int, detail: str, code: str = "VALIDATION_ERROR"):
        self.status_code, self.detail, self.code = status_code, detail, code


@app.exception_handler(ApiError)
async def api_error_handler(request: Request, exc: ApiError):
    return JSONResponse(status_code=exc.status_code, content={"detail": exc.detail, "code": exc.code})


@app.exception_handler(RequestValidationError)
async def request_validation_handler(request: Request, exc: RequestValidationError):
    return JSONResponse(
        status_code=422,
        content={"detail": "Нужны два файла: sales и stock (multipart/form-data)", "code": "VALIDATION_ERROR"},
    )


def csv_response(df: pd.DataFrame, filename: str) -> StreamingResponse:
    output = io.StringIO()
    df.to_csv(output, index=False)
    return StreamingResponse(
        iter([output.getvalue()]),
        media_type="text/csv",
        headers={"Content-Disposition": f"attachment; filename={filename}"},
    )


def cached_forecast() -> pd.DataFrame:
    try:
        return predictor.get_cached_forecast()
    except NotReadyError as e:
        raise ApiError(409, str(e), code="FORECAST_NOT_READY")


@app.get("/health")
async def health():
    return {"status": "ok"}


@app.post("/predict")
async def predict_post(sales: bytes = File(...), stock: bytes = File(...)):
    """Принимает два CSV (sales + stock) и возвращает CSV с прогнозом на 9 недель."""
    if not sales.strip() or not stock.strip():
        raise ApiError(400, "Пустой файл: нужны непустые sales и stock")
    try:
        df_sales = pd.read_csv(io.BytesIO(sales))
        df_stock = pd.read_csv(io.BytesIO(stock))
    except Exception as e:
        raise ApiError(400, f"Ошибка парсинга CSV: {e}")
    if df_sales.empty or df_stock.empty:
        raise ApiError(400, "Пустой файл: в sales или stock нет строк с данными")

    errors = validate_sales_csv(df_sales) + validate_stock_csv(df_stock)
    if errors:
        raise ApiError(422, "; ".join(errors))

    df_forecast = predictor.predict(df_sales, df_stock)
    return csv_response(df_forecast, "forecast.csv")


@app.get("/predict")
async def predict_get(
    article: Optional[str] = Query(None, description="Артикул товара"),
    warehouse: Optional[str] = Query(None, description="Наименование склада"),
):
    """Прогноз по артикулу и/или складу (CSV, 9 недель). Хотя бы один параметр обязателен."""
    if not article and not warehouse:
        raise ApiError(400, "Укажите хотя бы один параметр: article или warehouse")

    df_forecast = cached_forecast()
    mask = pd.Series(True, index=df_forecast.index)
    if article:
        mask &= df_forecast["article"] == article
    if warehouse:
        mask &= df_forecast["warehouse_name"] == warehouse
    return csv_response(df_forecast[mask], "filtered_forecast.csv")


@app.get("/predict/all")
async def predict_all(format: str = Query("csv", description="Формат ответа: csv или json")):
    """Весь прогноз целиком на 9 недель (79 101 строка при полном произведении 11 x 799)."""
    if format not in ("csv", "json"):
        raise ApiError(400, "Параметр format должен быть 'csv' или 'json'")
    df_forecast = cached_forecast()
    if format == "csv":
        return csv_response(df_forecast, "full_forecast.csv")
    return JSONResponse(content=df_forecast.to_dict(orient="records"))
''',
    "Dockerfile": r'''FROM python:3.11-slim

# Системные зависимости (libgomp1 нужен LightGBM)
RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential libgomp1 \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app
ENV PYTHONPATH=/app \
    PYTHONUNBUFFERED=1

# Python-зависимости
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Приложение, код признаков и модель
COPY app/ ./app/
COPY src/ ./src/
COPY model/ ./model/

# Модель загружается в память при старте контейнера
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
''',
    "tests/test_api.sh": r'''#!/bin/bash
# Самопроверка API. Запустите после поднятия контейнера.
# Usage: bash tests/test_api.sh [BASE_URL]

set -e
BASE_URL="${1:-http://localhost:8000}"

echo "=== 1. Health check ==="
curl -sf "$BASE_URL/health" | jq .
echo "OK"
echo

echo "=== 2. POST /predict ==="
curl -sf -X POST "$BASE_URL/predict" \
  -F "sales=@data/sample_sales.csv" \
  -F "stock=@data/sample_stock.csv" \
  -o /tmp/forecast.csv
echo "Получен файл: $(wc -l < /tmp/forecast.csv) строк"
head -3 /tmp/forecast.csv
echo "OK"
echo

ARTICLE=$(sed -n '2p' /tmp/forecast.csv | cut -d, -f3)
WAREHOUSE=$(sed -n '2p' /tmp/forecast.csv | cut -d, -f2)

echo "=== 3. GET /predict?article= ==="
curl -sf "$BASE_URL/predict?article=$ARTICLE" | head -3
echo "OK"
echo

echo "=== 4. GET /predict?warehouse= ==="
curl -sf "$BASE_URL/predict?warehouse=$WAREHOUSE" | head -3
echo "OK"
echo

echo "=== 5. GET /predict?article=&warehouse= ==="
curl -sf "$BASE_URL/predict?article=$ARTICLE&warehouse=$WAREHOUSE" | head -3
echo "OK"
echo

echo "=== 6. GET /predict/all?format=csv ==="
curl -sf "$BASE_URL/predict/all?format=csv" -o /tmp/full_forecast.csv
echo "Получен файл: $(wc -l < /tmp/full_forecast.csv) строк"
echo "OK"
echo

echo "=== 7. GET /predict/all?format=json ==="
curl -sf "$BASE_URL/predict/all?format=json" | jq '.[0:3]'
echo "OK"
echo

echo "=== 8. Error: GET /predict без параметров ==="
CODE=$(curl -s -o /dev/null -w "%{http_code}" "$BASE_URL/predict")
echo "HTTP $CODE (ожидается 400)"
[ "$CODE" = "400" ]
echo

echo "=== Все тесты пройдены! ==="
''',
    ".dockerignore": r'''.git
.gitignore
.ipynb_checkpoints/
.venv/
__pycache__/
*.pyc
notebooks/
docs/
data/
tests/
*.csv
''',
}

for rel, text in FILES.items():
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8", newline="\n")
    print("записан:", rel)

for rel in ("src/__init__.py", "app/__init__.py"):
    (ROOT / rel).write_text("", encoding="utf-8")


def pin(package):
    """Версия пакета из твоего окружения (чтобы контейнер собирался с теми же версиями)."""
    try:
        return f"{package}=={metadata.version(package)}"
    except metadata.PackageNotFoundError:
        return package


requirements = "\n".join(pin(p) for p in
                         ["lightgbm", "pandas", "numpy", "fastapi", "uvicorn", "python-multipart"]) + "\n"
(ROOT / "requirements.txt").write_text(requirements, encoding="utf-8")
print("\nrequirements.txt:\n" + requirements)


записан: src/features.py
записан: src/predictor_core.py
записан: src/raw_to_arrays.py
записан: app/predictor.py
записан: app/validators.py
записан: app/main.py
записан: Dockerfile
записан: tests/test_api.sh
записан: .dockerignore

requirements.txt:
lightgbm==4.7.0
pandas==3.0.6
numpy==2.4.6
fastapi==0.162.0
uvicorn==0.54.0
python-multipart==0.0.32



In [14]:
# ЭТАП W9. Сохраняем модель и справочники для контейнера и проверяем предсказание «как в контейнере»
import json
import sys

sys.path.insert(0, str(ROOT))
from app.predictor import Predictor

USE_MEDIAN = False      # True: в контейнер идёт медианная модель из W6 (если ячейка W6 запускалась)

if USE_MEDIAN:
    container_model, c_thr, c_scale, expected = model_med, MEDIAN_THRESHOLD, MEDIAN_SCALE, sub_med
else:
    container_model, c_thr, c_scale, expected = final_model, FINAL_THRESHOLD, FINAL_SCALE, sub

# --- 1. Модель и справочники в model/ ---
model_dir = ROOT / "model"
model_dir.mkdir(exist_ok=True)
# пишем модель текстом с переводами строк \n (на Windows save_model/Git могут дать CRLF, тогда LightGBM в Linux не читает файл)
(model_dir / "model.txt").write_text(container_model.booster_.model_to_string(), encoding="utf-8", newline="\n")
cfg = {
    "kind": "lightgbm",
    "weeks": WEEKS,
    "window": 180,
    "threshold": c_thr,
    "scale": c_scale,
    "warehouses": [str(x) for x in meta["warehouse_name"].cat.categories],    # порядок = коды складов
    "categories": [str(x) for x in pd.Categorical(meta["cat1"]).categories],   # порядок = коды категорий
    "article_category": {str(a): str(cat) for a, cat in art_cat.items()},
    "pairs": [[str(w), str(a)] for w, a in zip(meta["warehouse_name"], meta["article"])],   # серии модели
    "seed": SEED,
}
(model_dir / "config.json").write_text(json.dumps(cfg, ensure_ascii=False), encoding="utf-8")
print(f"Сохранено в model/: model.txt ({(model_dir / 'model.txt').stat().st_size / 1e6:.1f} МБ), "
      f"config.json ({(model_dir / 'config.json').stat().st_size / 1e3:.0f} КБ)")

# --- 2. Имитация сырых файлов, как их пришлёт API: последние 190 дней, несколько строк на день ---
LAST_DAYS = 190
c = D - 1
lo = c - LAST_DAYS + 1
wh_names = meta["warehouse_name"].astype(str).values
art_names = meta["article"].astype(str).values
cat_names = meta["cat1"].astype(str).values

si, di = np.nonzero(Q0[:, lo:c + 1] > 0)                       # продажи есть только в дни с продажами
qty = Q0[si, lo + di].astype(float)
base = pd.DataFrame({
    "date": dates[lo + di].strftime("%Y-%m-%d"), "warehouse_name": wh_names[si], "article": art_names[si],
    "sales_sum": qty * 100.0, "discount_sum": 0.0, "cities_name": "city_1",
    "category_lvl_1": cat_names[si], "category_lvl_2": "unknown", "category_lvl_3": "unknown",
    "characteristic": "unknown",
})
part1, part2 = base.copy(), base.copy()                         # одна продажа = две строки (два подразделения)
part1["quantity"], part1["composite_division"] = qty / 2, "division_1"
part2["quantity"], part2["composite_division"] = qty / 2, "division_2"
sales_raw = pd.concat([part1, part2], ignore_index=True)

ki, kd = np.nonzero(~np.isnan(ST[:, lo:c + 1]))                 # остаток есть не во все дни
val = ST[ki, lo + kd].astype(float)
stock_base = pd.DataFrame({"date": dates[lo + kd].strftime("%Y-%m-%d"),
                           "warehouse_name": wh_names[ki], "char_article": art_names[ki]})
s1, s2 = stock_base.copy(), stock_base.copy()                   # остаток разбит на две строки (дубли суммируются)
s1["stock"], s2["stock"] = val + 3.0, -3.0
stock_raw = pd.concat([s1, s2], ignore_index=True)
print(f"Сырые данные: sales {len(sales_raw):,} строк, stock {len(stock_raw):,} строк")

# --- 3. Прогноз контейнерным кодом и сравнение с прогнозом из ноутбука ---
predictor = Predictor.load(model_dir)
fc = predictor.predict(sales_raw, stock_raw)
fc_again = predictor.predict(sales_raw, stock_raw)
merged = fc.merge(expected, on=["week", "warehouse_name", "article"], suffixes=("_api", "_nb"))
diff = float((merged["quantity_api"] - merged["quantity_nb"]).abs().max())
same_rows = len(merged) == len(expected) == len(fc)
print(f"\nСтрок в прогнозе контейнера: {len(fc):,}; первая неделя: {fc['week'].iloc[0]}")
print(f"Максимальное расхождение с прогнозом из ноутбука: {diff:.4f}")
print("ПАРИТЕТ (сырые файлы -> контейнерный код = прогноз из ноутбука):",
      "ДА" if same_rows and diff < 1e-3 else "НЕТ")
print("Детерминизм (два запроса дают одинаковый результат):", "ДА" if fc.equals(fc_again) else "НЕТ")


Сохранено в model/: model.txt (2.8 МБ), config.json (401 КБ)
Сырые данные: sales 569,448 строк, stock 2,953,254 строк

Строк в прогнозе контейнера: 79,101; первая неделя: 2026-07-27
Максимальное расхождение с прогнозом из ноутбука: 0.0000
ПАРИТЕТ (сырые файлы -> контейнерный код = прогноз из ноутбука): ДА
Детерминизм (два запроса дают одинаковый результат): ДА


In [15]:
# ЭТАП W10. Примеры входных файлов (data/sample_*.csv) и проверка всех эндпоинтов API без Docker
import importlib
import io
import os
import sys

os.chdir(ROOT)                          # приложение ищет модель в папке model/ относительно текущей папки
sys.path.insert(0, str(ROOT))
from fastapi.testclient import TestClient
import app.main as main_module
importlib.reload(main_module)           # чистый старт: кэш прогноза пустой

# --- 1. Примеры входных файлов: 6 самых продаваемых пар за ~190 дней (меньше 180 дней API не примет) ---
top = np.argsort(Q0[:, c - LAST_DAYS + 1:c + 1].sum(axis=1))[::-1][:6]
sample_pairs = pd.DataFrame({"warehouse_name": wh_names[top], "article": art_names[top]})
first_day, last_day = dates[c - LAST_DAYS + 1].strftime("%Y-%m-%d"), dates[c].strftime("%Y-%m-%d")

real_sales, real_stock = ROOT / "data" / "raw" / "sales_dataset_hakaton.csv", ROOT / "data" / "raw" / "stock_dataset_hakaton.csv"
if real_sales.exists() and real_stock.exists():
    s_part = pd.read_csv(real_sales).merge(sample_pairs, on=["warehouse_name", "article"])
    stock_file = pd.read_csv(real_stock)
    if "char_article" not in stock_file.columns:
        stock_file = stock_file.rename(columns={"article": "char_article"})                # формат ТЗ: char_article
    k_part = stock_file.merge(sample_pairs.rename(columns={"article": "char_article"}),
                              on=["warehouse_name", "char_article"])
    source = "настоящие файлы из data/raw"
else:
    s_part = sales_raw.merge(sample_pairs, on=["warehouse_name", "article"])           # из ячейки W9
    k_part = stock_raw.merge(sample_pairs.rename(columns={"article": "char_article"}),
                             on=["warehouse_name", "char_article"])
    source = "данные, восстановленные в W9 (настоящих файлов в data/raw нет)"
s_part = s_part[(s_part["date"] >= first_day) & (s_part["date"] <= last_day)]
k_part = k_part[(k_part["date"] >= first_day) & (k_part["date"] <= last_day)]
(ROOT / "data").mkdir(exist_ok=True)
s_part.to_csv(ROOT / "data" / "sample_sales.csv", index=False)
k_part.to_csv(ROOT / "data" / "sample_stock.csv", index=False)
span = lambda d: (pd.to_datetime(d["date"]).max() - pd.to_datetime(d["date"]).min()).days + 1
print(f"Примеры сохранены ({source}): sample_sales {len(s_part):,} строк / {span(s_part)} дн., "
      f"sample_stock {len(k_part):,} строк / {span(k_part)} дн.")

# --- 2. Проверки эндпоинтов ---
client = TestClient(main_module.app)
sales_bytes = (ROOT / "data" / "sample_sales.csv").read_bytes()
stock_bytes = (ROOT / "data" / "sample_stock.csv").read_bytes()
files = lambda s, k: {"sales": ("sales.csv", s, "text/csv"), "stock": ("stock.csv", k, "text/csv")}
frame = lambda df: df.to_csv(index=False).encode("utf-8")
results = []


def check(name, ok, info=""):
    results.append({"проверка": name, "результат": "ок" if ok else "ОШИБКА", "детали": info})


def is_error(r, status):
    body = r.json() if r.headers.get("content-type", "").startswith("application/json") else {}
    return r.status_code == status and set(body) == {"detail", "code"}


r = client.get("/health")
check("GET /health", r.status_code == 200 and r.json() == {"status": "ok"})
r = client.get("/predict", params={"article": art_names[0]})
check("GET /predict до первого POST -> понятная ошибка 409", is_error(r, 409), f"HTTP {r.status_code}")

r = client.post("/predict", files=files(sales_bytes, stock_bytes))
fc_api = pd.read_csv(io.StringIO(r.text))
n_pairs = len(main_module.predictor.art["cfg"]["pairs"])
check("POST /predict: CSV нужного формата", r.status_code == 200 and r.headers["content-type"].startswith("text/csv")
      and list(fc_api.columns) == ["week", "warehouse_name", "article", "quantity"] and len(fc_api) == n_pairs * 9,
      f"HTTP {r.status_code}, строк {len(fc_api):,}, первая неделя {fc_api['week'].iloc[0]}")
check("POST /predict: нет NaN и отрицательных", fc_api["quantity"].notna().all() and (fc_api["quantity"] >= 0).all())
r_again = client.post("/predict", files=files(sales_bytes, stock_bytes))
check("POST /predict: тот же ответ при повторном запросе", r_again.text == r.text)

a, w = fc_api["article"].iloc[0], fc_api["warehouse_name"].iloc[0]
for name, params, expected_rows in [
    ("GET /predict?article=", {"article": a}, int((fc_api["article"] == a).sum())),
    ("GET /predict?warehouse=", {"warehouse": w}, int((fc_api["warehouse_name"] == w).sum())),
    ("GET /predict?article=&warehouse=", {"article": a, "warehouse": w}, 9),
]:
    rr = client.get("/predict", params=params)
    got = pd.read_csv(io.StringIO(rr.text))
    check(name, rr.status_code == 200 and len(got) == expected_rows, f"строк {len(got)} (ожидалось {expected_rows})")

rr = client.get("/predict/all", params={"format": "csv"})
check("GET /predict/all?format=csv", rr.status_code == 200 and len(pd.read_csv(io.StringIO(rr.text))) == n_pairs * 9)
rr = client.get("/predict/all", params={"format": "json"})
check("GET /predict/all?format=json", rr.status_code == 200 and len(rr.json()) == n_pairs * 9
      and set(rr.json()[0]) == {"week", "warehouse_name", "article", "quantity"})

check("GET /predict без параметров -> 400", is_error(client.get("/predict"), 400))
check("GET /predict/all?format=xml -> 400", is_error(client.get("/predict/all", params={"format": "xml"}), 400))

s_df, k_df = pd.read_csv(io.BytesIO(sales_bytes)), pd.read_csv(io.BytesIO(stock_bytes))
check("пустые файлы -> 400", is_error(client.post("/predict", files=files(b"", b"")), 400))
check("CSV только с заголовком -> 400",
      is_error(client.post("/predict", files=files(frame(s_df.head(0)), stock_bytes)), 400))
check("нет колонки quantity -> 422",
      is_error(client.post("/predict", files=files(frame(s_df.drop(columns=["quantity"])), stock_bytes)), 422))
short = s_df[pd.to_datetime(s_df["date"]) >= pd.to_datetime(s_df["date"]).max() - pd.Timedelta(days=100)]
check("история меньше 180 дней -> 422", is_error(client.post("/predict", files=files(frame(short), stock_bytes)), 422))
neg = s_df.copy()
neg.loc[neg.index[0], "quantity"] = -1
check("отрицательный quantity -> 422", is_error(client.post("/predict", files=files(frame(neg), stock_bytes)), 422))
neg_stock = k_df.copy()
neg_stock.loc[neg_stock.index[:5], "stock"] = -5
check("отрицательный stock допустим -> 200",
      client.post("/predict", files=files(sales_bytes, frame(neg_stock))).status_code == 200)
bad_date = s_df.copy()
bad_date["date"] = bad_date["date"].str.replace("-", "/")
check("неверный формат даты -> 422", is_error(client.post("/predict", files=files(frame(bad_date), stock_bytes)), 422))
check("не передан файл stock -> 422",
      is_error(client.post("/predict", files={"sales": ("sales.csv", sales_bytes, "text/csv")}), 422))

report = pd.DataFrame(results)
print(f"\nПройдено: {(report['результат'] == 'ок').sum()} из {len(report)}")
display(report)


C:\Users\nikit\anaconda3\envs\forecast\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Примеры сохранены (настоящие файлы из data/raw): sample_sales 13,272 строк / 190 дн., sample_stock 3,608 строк / 190 дн.

Пройдено: 20 из 20


,проверка,результат,детали
0,GET /health,ок,
1,GET /predict до первого POST -> понятная ошибк...,ок,HTTP 409
2,POST /predict: CSV нужного формата,ок,"HTTP 200, строк 79,101, первая неделя 2026-07-27"
3,POST /predict: нет NaN и отрицательных,ок,
4,POST /predict: тот же ответ при повторном запросе,ок,
5,GET /predict?article=,ок,строк 99 (ожидалось 99)
6,GET /predict?warehouse=,ок,строк 7191 (ожидалось 7191)
7,GET /predict?article=&warehouse=,ок,строк 9 (ожидалось 9)
8,GET /predict/all?format=csv,ок,
9,GET /predict/all?format=json,ок,


In [16]:
# ЭТАП W11. Проверка на НАСТОЯЩИХ сырых файлах (запускать после W8-W9, нужны data/raw/*.csv)
# 1) влияет ли drop_duplicates в остатках на значения; 2) паритет «сырые файлы -> контейнер = прогноз из ноутбука»
import sys
sys.path.insert(0, str(ROOT))
import importlib
import app.predictor, src.raw_to_arrays
importlib.reload(src.raw_to_arrays); importlib.reload(app.predictor)
from app.predictor import Predictor

raw_sales_p = ROOT / "data" / "raw" / "sales_dataset_hakaton.csv"
raw_stock_p = ROOT / "data" / "raw" / "stock_dataset_hakaton.csv"
assert raw_sales_p.exists() and raw_stock_p.exists(), "Положи настоящие sales/stock в data/raw/"
sales_real = pd.read_csv(raw_sales_p)
stock_real = pd.read_csv(raw_stock_p)
stock_real.columns = [str(c).strip().lstrip("\ufeff") for c in stock_real.columns]    # лишние пробелы и BOM в названиях
print("Колонки stock:", stock_real.columns.tolist())
print("Колонки sales:", sales_real.columns.tolist())
if "char_article" not in stock_real.columns and "article" in stock_real.columns:
    stock_real = stock_real.rename(columns={"article": "char_article"})                  # по ТЗ колонка называется char_article
missing = {"date", "warehouse_name", "char_article", "stock"} - set(stock_real.columns)
assert not missing, f"В stock нет колонок {sorted(missing)}: проверь, тот ли это файл (формат из ТЗ, раздел 2)"
stock_real = stock_real[pd.to_datetime(stock_real["date"]) <= CUT_DATE]
sales_real = sales_real[pd.to_datetime(sales_real["date"]) <= CUT_DATE]

# --- 1. Дубли остатков: ТЗ требует суммировать ВСЕ строки с одинаковым (date, warehouse_name, char_article) ---
key = ["date", "warehouse_name", "char_article"]
sum_all = stock_real.groupby(key)["stock"].sum()
dedup = stock_real.drop_duplicates(subset=key + ["stock"])
sum_dedup = dedup.groupby(key)["stock"].sum()
n_dup_rows = len(stock_real) - len(dedup)
n_cells_diff = int((sum_all.reindex(sum_dedup.index) != sum_dedup).sum())
print(f"Строк stock: {len(stock_real):,}; полных дублей (ключ + stock): {n_dup_rows:,}; "
      f"ячеек, где значение отличается из-за drop_duplicates: {n_cells_diff:,}")
print("-> drop_duplicates НЕ нужен: ТЗ говорит, что строки с одним ключом - разные записи, их суммируют"
      if n_dup_rows else "-> полных дублей нет, оба способа дают одно и то же")

# --- 2. Паритет на настоящих файлах за последние 190 дней ---
# Контейнер принимает и более длинную историю (ТЗ: "данных может быть больше полугода"), поэтому отдаём всю историю:
# так остаток у серий без записей в последние месяцы тянется вперёд так же, как в base_grid.
s_in, k_in = sales_real, stock_real
pred = Predictor.load(ROOT / "model")
fc = pred.predict(s_in, k_in)
expected = sub                                   # прогноз из ноутбука (W4), USE_MEDIAN=False
m = fc.merge(expected, on=["week", "warehouse_name", "article"], suffixes=("_api", "_nb"))
diff = (m["quantity_api"] - m["quantity_nb"]).abs()
print(f"\nСтрок: контейнер {len(fc):,}, ноутбук {len(expected):,}, совпало по ключу {len(m):,}")
print(f"Максимальное расхождение: {diff.max():.4f}; строк с расхождением > 1e-3: {(diff > 1e-3).sum():,}")
print("ПАРИТЕТ НА НАСТОЯЩИХ ФАЙЛАХ:", "ДА" if len(m) == len(expected) == len(fc) and diff.max() < 1e-3 else "НЕТ")
if diff.max() >= 1e-3:
    bad = m.loc[diff > 1e-3, ["warehouse_name", "article"]].drop_duplicates()
    print(f"Серий с расхождением: {len(bad)}; примеры:"); display(bad.head())


Колонки stock: ['date', 'warehouse_name', 'article', 'stock']
Колонки sales: ['date', 'composite_division', 'cities_name', 'warehouse_name', 'category_lvl_1', 'category_lvl_2', 'category_lvl_3', 'article', 'characteristic', 'quantity', 'sales_sum', 'discount_sum']
Строк stock: 6,092,305; полных дублей (ключ + stock): 420,999; ячеек, где значение отличается из-за drop_duplicates: 15,712
-> drop_duplicates НЕ нужен: ТЗ говорит, что строки с одним ключом - разные записи, их суммируют

Строк: контейнер 79,101, ноутбук 79,101, совпало по ключу 79,101
Максимальное расхождение: 0.0000; строк с расхождением > 1e-3: 0
ПАРИТЕТ НА НАСТОЯЩИХ ФАЙЛАХ: ДА
